In [1]:
import pandas as pd
from datetime import date, timedelta, datetime
import os, shutil, pathlib, glob, time
from nbconvert import PythonExporter
import polars as pl
from IPython.display import display

# ── FIX: định nghĩa convert_notebook dùng PythonExporter ──────────────────
def convert_notebook(nb_path: str) -> str:
    """Convert .ipynb → Python source string via nbconvert."""
    exporter = PythonExporter()
    source, _ = exporter.from_filename(nb_path)
    return source


In [2]:
RUN_NOTEBOOKS = True

notebook_dict = {
    'IEX_optimized.ipynb'    : 1,
    'RTA_optimized (1).ipynb': 1,
}

PRODUCTIVE_THRESHOLD = 0.90

DATE_FROM = None
DATE_TO   = None

LOB_EXCLUDE_CONTAINS   = ['Support', 'Training']
SHIFT_TRACKING_EXCLUDE = [
    'Training Offline', 'Off Phone Misc', 'PO', 'Off', 'Offline',
    'SL', 'AL', 'CO', 'LWP', 'Termination', 'NCNS',
]


In [3]:
_now = datetime.now()
days_back = 2 if _now.hour < 6 else 1

if DATE_TO is None:
    _auto_to = _now.date() - timedelta(days=days_back)
else:
    _auto_to = datetime.strptime(DATE_TO, "%Y-%m-%d").date()

if DATE_FROM is None:
    _auto_from = _auto_to.replace(day=1)
else:
    _auto_from = datetime.strptime(DATE_FROM, "%Y-%m-%d").date()

start_date = str(_auto_from)
end_date   = str(_auto_to)

print(f"{_now.strftime('%Y-%m-%d %H:%M')} | days_back={days_back} | {start_date} → {end_date}")

first_glob = os.path.expanduser("~").replace("\\", "/")
test_path  = f"{first_glob}/Concentrix Corporation"
if not os.path.exists(test_path):
    raise FileNotFoundError(f"Root directory not found: {test_path}")

CODE_PATH = (f"{first_glob}/Concentrix Corporation/"
             "WFM-Expedia-HCM - Branding files/BI_Task/CODE")
RAW_PATH  = (f"{first_glob}/Concentrix Corporation/"
             "WFM-Expedia-HCM - Branding files/Rawdata")

folder_paths = {
    "output_iex_base": f"{RAW_PATH}/OUTPUT_AGENT_IEX_BASE",
    "storage_rta"    : f"{RAW_PATH}/STORAGE_OUTPUT_RTA",
    "python_code"    : f"{CODE_PATH}/Python_Code",
    "resources"      : f"{CODE_PATH}/Resources",
}

print(f"Base dir: {first_glob}")

def convert_to_datetime(struct_time):
    return datetime(*struct_time[:6])

def input_data_glob(data_dir):
    """Read all xlsx/csv files in a directory (recursive) into one DataFrame."""
    def read_file(filename):
        try:
            mtime = convert_to_datetime(time.localtime(os.path.getmtime(filename)))
            df = (pl.read_excel(filename, infer_schema_length=0)
                  if filename.suffix.lower() == ".xlsx"
                  else pl.read_csv(filename, infer_schema_length=0, encoding="utf-8"))
            return df.with_columns(pl.lit(filename.stem).alias("sheet_name"),
                                   pl.lit(mtime).alias("Export time"))
        except Exception:
            try:
                return (pl.read_csv(filename, infer_schema_length=0,
                                    encoding="ISO-8859-1", ignore_errors=True)
                        .with_columns(pl.lit(filename.stem).alias("sheet_name"),
                                      pl.lit(mtime).alias("Export time")))
            except Exception as e:
                print(f"  Skip {filename.name}: {e}")
                return None

    dfs = [df for f in pathlib.Path(data_dir).glob("**/*.*")
           if f.suffix.lower() in (".xlsx", ".csv")
           and (df := read_file(f)) is not None]
    if not dfs:
        return pl.DataFrame()
    col_dtypes = {col: {df[col].dtype for df in dfs if col in df.columns}
                  for col in set(c for df in dfs for c in df.columns)}
    dfs = [df.with_columns(pl.col(c).cast(pl.Utf8)
                           for c, t in col_dtypes.items()
                           if c in df.columns and len(t) > 1)
           for df in dfs]
    return pl.concat(dfs, how="vertical")

def input_data(folder_path, sheet_name=None):
    file_paths = glob.glob(f"{folder_path}/*.xlsx") + glob.glob(f"{folder_path}/*.csv")
    df_list = []
    for file in file_paths:
        try:
            if file.endswith(".xlsx"):
                df = pl.read_excel(file, sheet_name=sheet_name, infer_schema_length=0)
            else:
                try:
                    df = pl.read_csv(file, encoding="utf-8", infer_schema_length=0)
                except Exception:
                    df = pl.read_csv(file, encoding="ISO-8859-1",
                                     ignore_errors=True, infer_schema_length=0)
            df_list.append(df.with_columns(pl.col(c).cast(pl.String) for c in df.columns))
        except Exception as e:
            print(f"  Skip {os.path.basename(file)}: {e}")
    return pl.concat(df_list, how="vertical") if df_list else pl.DataFrame()

IEX_BASE = input_data_glob(folder_paths["output_iex_base"])
print(f"IEX_BASE loaded: {IEX_BASE.height:,} rows")


2026-09-02 22:13 | days_back=1 | 2026-09-01 → 2026-09-01
Base dir: C:/Users/huuchinh.nguyen
IEX_BASE loaded: 109,247 rows


In [4]:
# ── FIX: capture path trước loop, dùng shallow-copy globals để tránh
# sub-notebook ghi đè biến cha (đặc biệt folder_paths) ────────────────────
if not RUN_NOTEBOOKS:
    print("[SKIP ALL] RUN_NOTEBOOKS = False")
else:
    _nb_dir = folder_paths['python_code']  # snapshot trước khi exec có thể overwrite
    for nbook, flag in notebook_dict.items():
        if flag == 0:
            print(f"[SKIP] {nbook}")
            continue
        nbook_path = f"{_nb_dir}/{nbook}"
        if os.path.isfile(nbook_path):
            print(f"[RUN]  {nbook}")
            try:
                # {**globals()} = shallow copy → sub-notebook đọc được biến cha
                # nhưng KHÔNG ghi đè lại namespace cha
                exec(convert_notebook(nbook_path), {**globals()})
                print(f"[DONE] {nbook}")
            except Exception as e:
                print(f"[ERROR] {nbook}: {e}")
        else:
            print(f"[NOT FOUND] {nbook_path}")


[RUN]  IEX_optimized.ipynb
--- FULL FOLDER PATHS LIST ---
input_iex: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/INPUT_AGENT_IEX_FOR_REPORT
input_hc_master: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Master Database - 2026.xlsx
output_iex: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_AGENT_IEX_FOR_REPORT
output_iex_all: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/STORAGE_OUTPUT_AGENT_IEX
output_iex_intervals: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_AGENT_IEX_INTERVALS
output_iex_base: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_AGENT_IEX_BASE
hc_extend_by_month: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Extend by Month
input_iex_schedule_ad

<string>:257: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
<string>:291: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.


✅ Exported 15 file(s) to 'C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_AGENT_IEX_BASE':
   IEX_Base_2026_03_09.csv  (2 rows)
   IEX_Base_2026_03_16.csv  (13 rows)
   IEX_Base_2026_03_23.csv  (4 rows)
   IEX_Base_2026_03_30.csv  (3 rows)
   IEX_Base_2026_04_27.csv  (1 rows)
   IEX_Base_2026_06_15.csv  (1 rows)
   IEX_Base_2026_06_22.csv  (13 rows)
   IEX_Base_2026_07_06.csv  (4 rows)
   IEX_Base_2026_07_20.csv  (5097 rows)
   IEX_Base_2026_07_27.csv  (4917 rows)
   IEX_Base_2026_08_03.csv  (4873 rows)
   IEX_Base_2026_08_10.csv  (4472 rows)
   IEX_Base_2026_08_17.csv  (4168 rows)
   IEX_Base_2026_08_24.csv  (3750 rows)
   IEX_Base_2026_08_31.csv  (7466 rows)
Valid weeks found: ['2026-07-20', '2026-07-27', '2026-08-03', '2026-08-10', '2026-08-17', '2026-08-24', '2026-08-31']


<string>:342: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.


Rows after filter: 34743
Weeks retained: ['2026-07-20', '2026-07-27', '2026-08-03', '2026-08-10', '2026-08-17', '2026-08-24', '2026-08-31']


...


=== STAGE 4: IEX_EXTEND rows (final output) ===


,Date,IEX_ID,Datetime_First_Start_Shift,Datetime_First_End_Shift,First Shift,Open Time,Extra Time,Target,Time_Of_Day,Shift Tracking


Valid weeks: [Timestamp('2026-07-20 00:00:00'), Timestamp('2026-07-27 00:00:00'), Timestamp('2026-08-03 00:00:00'), Timestamp('2026-08-10 00:00:00'), Timestamp('2026-08-17 00:00:00'), Timestamp('2026-08-24 00:00:00'), Timestamp('2026-08-31 00:00:00')]
[DONE] IEX_optimized.ipynb
[RUN]  RTA_optimized (1).ipynb
--- FULL FOLDER PATHS LIST ---
input_iex_rawdata: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/INPUT_AGENT_IEX_FOR_REPORT
storage_iex_rawdata: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/STORAGE_INPUT_AGENT_IEX
input_agent_activity: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/INPUT_AGENT_ACTIVITY_FOR_REPORT
storage_agent_activity: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/STORAGE_INPUT_AGENT_ACTIVITY
input_iex: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPU

<string>:379: DeprecationWarning: `DataFrame.melt` is deprecated. Use `unpivot` instead, with `index` instead of `id_vars` and `on` instead of `value_vars`


shape: (2_376, 7)
┌───────────┬────────────┬─────────────────┬───────┬───────────────┬──────────────┬─────────────┐
│ EID       ┆ Date       ┆ ot_type         ┆ hours ┆ nsa_authorize ┆ ot_authorize ┆ ot_status   │
│ ---       ┆ ---        ┆ ---             ┆ ---   ┆ ---           ┆ ---          ┆ ---         │
│ str       ┆ date       ┆ str             ┆ f64   ┆ str           ┆ str          ┆ str         │
╞═══════════╪════════════╪═════════════════╪═══════╪═══════════════╪══════════════╪═════════════╡
│ 102458044 ┆ 2025-04-30 ┆ OT3.0X: 8.0 hrs ┆ 8.0   ┆               ┆ Authorized   ┆ (8.0 hrs -  │
│           ┆            ┆                 ┆       ┆               ┆              ┆ Authorized) │
│ 103173195 ┆ 2026-05-06 ┆ NSA: 7.0 hrs,   ┆ 2.0   ┆ Authorized    ┆ Authorized   ┆ (2.0 hrs -  │
│           ┆            ┆ OT1.5X: 2.0 hrs ┆       ┆               ┆              ┆ Authorized) │
│ 103293732 ┆ 2026-08-14 ┆ OT2.0X: 3.0 hrs ┆ 3.0   ┆               ┆ Authorized   ┆ (3.0 hrs -  │
│ 

In [5]:
pl.Config.set_tbl_rows(50)
pl.Config.set_tbl_cols(100)

RTA_REPORT_GENERATE = input_data(folder_paths["storage_rta"])
print("Columns:", RTA_REPORT_GENERATE.columns)

RTA_REPORT_GENERATE = RTA_REPORT_GENERATE.with_columns([
    pl.col("Date_Converted").str.slice(0, 10).str.strptime(pl.Date, format="%Y-%m-%d"),
    pl.col("Target").cast(pl.Float64),
    pl.col("duration").cast(pl.Float64),
    pl.col("sum_productive").cast(pl.Float64),
    pl.col("start").str.strip_chars().str.to_datetime("%Y-%m-%d %H:%M:%S%.3f", strict=False),
    pl.col("end").str.strip_chars().str.to_datetime("%Y-%m-%d %H:%M:%S%.3f", strict=False),
    pl.col("IEX ID").cast(pl.Utf8).str.strip_chars().str.strip_chars('"').alias("IEX ID"),
])

RTA_REPORT_GENERATE = RTA_REPORT_GENERATE.filter(
    pl.col("duration").is_not_null() | (pl.col("Target") > 0)
)

RTA_REPORT_GENERATE = RTA_REPORT_GENERATE.with_columns([
    pl.col("Shift Tracking").cast(pl.Utf8).str.strip_chars(),
    pl.col("Open Time").cast(pl.Float64).fill_null(0.0),
    pl.col("Extra Time").cast(pl.Float64).fill_null(0.0),
])

RTA_REPORT_GENERATE = RTA_REPORT_GENERATE.with_columns([
    pl.when(pl.col("Shift Tracking") == "HDL").then(0.5)
    .when(
        pl.col("Shift Tracking").is_in(["PR", "PR - OT", "PO"]) &
        ((pl.col("Open Time") > 0) | (pl.col("Extra Time") > 0))
    ).then(1.0)
    .when(
        pl.col("Shift Tracking").is_in([
            "Training Offline", "Sick Leave", "Paid Leave", "Billable Training"
        ]) & (pl.col("Open Time") == 0)
    ).then(1.0)
    .when(
        ~pl.col("Shift Tracking").is_in(["HDL", "PR"]) &
        (pl.col("Open Time") == 0)
    ).then(0.0)
    .otherwise(None)
    .alias("hc_present")
])

print(f"RTA_REPORT_GENERATE: {RTA_REPORT_GENERATE.height:,} rows")


Columns: ['Year', 'Month', 'Week_Monday', 'Date_Converted', 'Employee Name', 'Email Id', 'OracleID', 'IEX ID', 'Target', 'Datetime_Fluctuate_Start_Shift', 'Datetime_Fluctuate_End_Shift', 'Fluctuate Shift', 'Datetime_First_Start_Shift', 'Datetime_First_End_Shift', 'First Shift', 'Alias', 'LOB', 'LOB_2', 'LOB_3', 'Supervisor Name', 'Wave', 'Detail Status', 'Status', 'Time_Of_Day', 'Open Time', 'Extra Time', 'Break Time', 'Lunch Time', 'Training', 'NCNS', 'AL', 'Unplanned', 'Planned', 'Roster Presented', 'Roster Scheduled', 'Night_Shift', 'Shift Tracking', 'OT Type', 'Combined OT Range', 'OT PreShift', 'OT PostShift', 'start', 'end', 'total_time_chat_handle', 'sum_productive', 'break', 'lunch', 'coaching-idle', 'training-idle', 'outbound-idle', 'break_count', 'other_status', 'over_break', 'over_lunch', 'exceed_break', 'duration', 'hc_actual', 'hc_schedule', 'time_late', 'time_leave', 'lateness', 'adherence_time', 'ramco_marked', 'nsa_authorize', 'ot_authorize', 'hours', 'ot_status', 'ot_t

In [6]:
FLOAT_COLS = [
    "Target", "sum_productive", "duration", "hc_actual", "hc_schedule",
    "Open Time", "Extra Time", "Break Time", "Lunch Time", "Training",
    "Time_Of_Day", "NCNS", "AL",
    "break", "lunch", "over_break", "over_lunch", "exceed_break",
    "time_late", "time_leave", "adherence_time", "lateness",
    "total_time_chat_handle", "other_status", "training-idle",
    "coaching-idle", "outbound-idle", "break_count",
    "OT PreShift", "OT PostShift",
]
_avail_float = [c for c in FLOAT_COLS if c in RTA_REPORT_GENERATE.columns]

RTA_CAST = RTA_REPORT_GENERATE.with_columns([
    pl.col(c).cast(pl.Float64, strict=False).alias(c) for c in _avail_float
])

for _col in ["Datetime_Fluctuate_Start_Shift", "Datetime_Fluctuate_End_Shift",
             "Datetime_First_Start_Shift", "Datetime_First_End_Shift"]:
    if _col in RTA_CAST.columns:
        RTA_CAST = RTA_CAST.with_columns(
            pl.col(_col).str.strip_chars()
              .str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False)
              .alias(_col)
        )

RTA_CAST = RTA_CAST.with_columns([
    pl.when(pl.col("Target").is_not_null() & (pl.col("Target") > 0))
      .then((pl.col("sum_productive") / pl.col("Target") * 100).round(1))
      .otherwise(None)
      .alias("productive_pct"),

    pl.when(pl.col("start").is_not_null() & pl.col("end").is_not_null())
      .then(((pl.col("end") - pl.col("start")).dt.total_seconds() / 3600.0).round(2))
      .otherwise(None)
      .alias("actual_span_h"),

    (pl.col("Target") - pl.col("sum_productive")).round(2).alias("productive_gap_h"),

    pl.when(
        pl.col("Datetime_Fluctuate_Start_Shift").is_not_null() & pl.col("start").is_not_null()
    ).then(
        ((pl.col("start") - pl.col("Datetime_Fluctuate_Start_Shift"))
         .dt.total_seconds() / 60.0).round(1)
    ).otherwise(None).alias("late_vs_sched_min"),

    pl.when(
        pl.col("Datetime_Fluctuate_End_Shift").is_not_null() & pl.col("end").is_not_null()
    ).then(
        ((pl.col("Datetime_Fluctuate_End_Shift") - pl.col("end"))
         .dt.total_seconds() / 60.0).round(1)
    ).otherwise(None).alias("early_leave_min"),
])

low_prod = RTA_CAST.filter(
    pl.col("Target").is_not_null() &
    (pl.col("Target") > 0) &
    pl.col("sum_productive").is_not_null() &
    (pl.col("sum_productive") < pl.col("Target") * PRODUCTIVE_THRESHOLD)
).filter(
    (pl.col("Date_Converted") >= pl.lit(start_date).str.strptime(pl.Date, "%Y-%m-%d")) &
    (pl.col("Date_Converted") <= pl.lit(end_date).str.strptime(pl.Date, "%Y-%m-%d"))
)

if "LOB" in low_prod.columns:
    for _kw in LOB_EXCLUDE_CONTAINS:
        low_prod = low_prod.filter(
            pl.col("LOB").is_null() | ~pl.col("LOB").str.contains(_kw, literal=True)
        )

if "Shift Tracking" in low_prod.columns:
    low_prod = low_prod.filter(
        pl.col("Shift Tracking").is_null() |
        ~pl.col("Shift Tracking").is_in(SHIFT_TRACKING_EXCLUDE)
    )

if "start" in low_prod.columns:
    low_prod = low_prod.filter(pl.col("start").is_not_null())

low_prod = low_prod.sort(["Date_Converted", "productive_pct"])
print(f"Low productive rows: {low_prod.height:,}  "
      f"(< {int(PRODUCTIVE_THRESHOLD*100)}% | {start_date} → {end_date})")

DISPLAY_COLS = [
    "Date_Converted", "Week_Monday",
    "Employee Name", "Email Id", "IEX ID", "OracleID",
    "LOB", "LOB_2", "Detail Status", "Alias", "Supervisor Name",
    "First Shift", "Fluctuate Shift", "Shift Tracking",
    "Datetime_Fluctuate_Start_Shift", "Datetime_Fluctuate_End_Shift",
    "Target",
    "start", "end", "actual_span_h",
    "late_vs_sched_min", "early_leave_min",
    "sum_productive", "productive_pct", "productive_gap_h",
    "duration",
    "break", "over_break", "exceed_break", "break_count",
    "lunch", "over_lunch",
    "other_status", "training-idle", "coaching-idle", "outbound-idle",
]
_avail_disp = [c for c in DISPLAY_COLS if c in low_prod.columns]

_idle_cols = [c for c in ["training-idle", "coaching-idle", "other_status"] if c in low_prod.columns]
low_prod_display = (
    low_prod
    .with_columns([
        pl.coalesce([pl.col(c).cast(pl.Float64, strict=False) for c in _idle_cols])
          .fill_null(0.0)
          .alias("_idle_sum")
        if len(_idle_cols) == 1
        else
        sum(pl.col(c).cast(pl.Float64, strict=False).fill_null(0.0) for c in _idle_cols)
          .alias("_idle_sum")
    ])
    .with_columns([
        (pl.col("sum_productive").fill_null(0.0) + pl.col("_idle_sum"))
          .round(2)
          .alias("total_logged_h"),
        pl.when(pl.col("Target").is_not_null() & (pl.col("Target") > 0))
          .then(
              ((pl.col("sum_productive").fill_null(0.0) + pl.col("_idle_sum"))
               / pl.col("Target") * 100).round(1)
          )
          .otherwise(None)
          .alias("total_logged_pct"),
    ])
    .drop("_idle_sum")
    .select(_avail_disp + [c for c in ["total_logged_h", "total_logged_pct"] if c not in _avail_disp])
)

print(f"\n{'='*70}")
print(f"SUMMARY — Low Productive (< {int(PRODUCTIVE_THRESHOLD*100)}% | {start_date} → {end_date})")
print("="*70)
display(low_prod_display.select([
    pl.len().alias("total_rows"),
    pl.col("productive_pct").mean().round(1).alias("avg_productive_pct"),
    pl.col("productive_pct").min().round(1).alias("min_productive_pct"),
    pl.col("productive_gap_h").mean().round(2).alias("avg_gap_h"),
    pl.col("productive_gap_h").sum().round(2).alias("total_gap_h"),
]))

print("\nBy Shift Tracking:")
display(
    low_prod_display.group_by("Shift Tracking")
    .agg([pl.len().alias("count"),
          pl.col("productive_pct").mean().round(1).alias("avg_pct"),
          pl.col("productive_gap_h").mean().round(2).alias("avg_gap_h")])
    .sort("count", descending=True)
)

print("\nBy LOB:")
display(
    low_prod_display.group_by("LOB")
    .agg([pl.len().alias("count"),
          pl.col("productive_pct").mean().round(1).alias("avg_pct"),
          pl.col("productive_gap_h").mean().round(2).alias("avg_gap_h")])
    .sort("count", descending=True)
)

with pl.Config(tbl_rows=200, tbl_cols=60, fmt_str_lengths=40, tbl_width_chars=1000):
    display(low_prod_display)

_out = f"{folder_paths['resources']}/low_productive_filter.csv"
low_prod_display.write_csv(_out)
print(f"\nExported {low_prod_display.height:,} rows → {_out}")


Low productive rows: 0  (< 90% | 2026-09-01 → 2026-09-01)

SUMMARY — Low Productive (< 90% | 2026-09-01 → 2026-09-01)


total_rows,avg_productive_pct,min_productive_pct,avg_gap_h,total_gap_h
u32,f64,f64,f64,f64
0,null,null,null,0.0



By Shift Tracking:


Shift Tracking,count,avg_pct,avg_gap_h
str,u32,f64,f64



By LOB:


LOB,count,avg_pct,avg_gap_h
str,u32,f64,f64


Date_Converted,Week_Monday,Employee Name,Email Id,IEX ID,OracleID,LOB,LOB_2,Detail Status,Alias,Supervisor Name,First Shift,Fluctuate Shift,Shift Tracking,Datetime_Fluctuate_Start_Shift,Datetime_Fluctuate_End_Shift,Target,start,end,actual_span_h,late_vs_sched_min,early_leave_min,sum_productive,productive_pct,productive_gap_h,duration,break,over_break,exceed_break,break_count,lunch,over_lunch,other_status,training-idle,coaching-idle,outbound-idle,total_logged_h,total_logged_pct
date,str,str,str,str,str,str,str,str,str,str,str,str,str,datetime[μs],datetime[μs],f64,datetime[ms],datetime[ms],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64



Exported 0 rows → C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Resources/low_productive_filter.csv
